In [1]:
import pandas as pd
import numpy as np

DATA_PATH = "/content/drive/MyDrive/Dataset (Decode Labs)/Dataset for Data Analytics.xlsx"  # update this path

df = pd.read_excel(DATA_PATH)
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [2]:
df.info()
print("\nShape:", df.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

In [3]:
missing_summary = df.isna().sum().to_frame("missing_count")
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(df) * 100).round(2)
missing_summary[missing_summary["missing_count"] > 0]

,missing_count,missing_pct
CouponCode,309,25.75


In [4]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import LabelEncoder

df_impute = df.copy()

le = LabelEncoder()
known_mask = df_impute["CouponCode"].notna()
le.fit(df_impute.loc[known_mask, "CouponCode"])

encoded_coupon = pd.Series(np.nan, index=df_impute.index)
encoded_coupon[known_mask] = le.transform(df_impute.loc[known_mask, "CouponCode"])

knn_features = df_impute[["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]].copy()
knn_features["CouponCode_enc"] = encoded_coupon.values

imputer = KNNImputer(n_neighbors=5)
imputed_array = imputer.fit_transform(knn_features)

imputed_coupon_codes = np.round(imputed_array[:, -1]).astype(int)
imputed_coupon_codes = np.clip(imputed_coupon_codes, 0, len(le.classes_) - 1)

df_impute["CouponCode"] = le.inverse_transform(imputed_coupon_codes)

print("Missing values remaining:", df_impute["CouponCode"].isna().sum())
df_impute["CouponCode"].value_counts()

Missing values remaining: 0


,count
CouponCode,
SAVE10,544
FREESHIP,345
WINTER15,311


In [5]:
df_clean = df_impute.copy()
numeric_cols = ["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]

outlier_report = {}

for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    n_outliers = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    outlier_report[col] = {"lower_bound": lower, "upper_bound": upper, "outliers_found": n_outliers}

    df_clean[col] = np.clip(df_clean[col], lower, upper)

pd.DataFrame(outlier_report).T

,lower_bound,upper_bound,outliers_found
Quantity,-1.00000,7.00000,0.0
UnitPrice,-317.19875,1024.83125,0.0
ItemsInCart,-0.50000,11.50000,0.0
TotalPrice,-1341.41250,3330.40750,8.0


In [6]:
# Feature 1: Order month (seasonality)
df_clean["OrderMonth"] = df_clean["Date"].dt.month

# Feature 2: Discount percentage applied
expected_price = df_clean["Quantity"] * df_clean["UnitPrice"]
df_clean["DiscountPct"] = ((expected_price - df_clean["TotalPrice"]) / expected_price * 100).round(2)

# Feature 3: Repeat customer flag
customer_order_counts = df_clean["CustomerID"].value_counts()
df_clean["IsRepeatCustomer"] = df_clean["CustomerID"].map(customer_order_counts).gt(1).astype(int)

df_clean[["OrderMonth", "DiscountPct", "IsRepeatCustomer"]].head()

,OrderMonth,DiscountPct,IsRepeatCustomer
0,1,0.0,0
1,8,0.0,0
2,2,-0.0,0
3,10,0.0,0
4,5,0.0,0


In [7]:
print("Final shape:", df_clean.shape)
df_clean.head()

Final shape: (1200, 17)


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice,OrderMonth,DiscountPct,IsRepeatCustomer
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10,1,0.0,0
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70,8,0.0,0
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40,2,-0.0,0
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19,10,0.0,0
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04,5,0.0,0


In [8]:
df_clean.to_csv("/content/drive/MyDrive/cleaned_dataset_project1.csv", index=False)
print("Saved to Google Drive")

Saved to Google Drive
